## Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from textblob import TextBlob
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# Load the pre-processed analytic dataset
# (This dataset was generated by the data cleaning & sentiment pipeline)
df = pd.read_csv('data/processed/Final_Analytic_Dataset.csv')
print("Dataset loaded:", df.shape)
print("Columns:", df.columns.tolist())
print()
print(df['App_Name'].value_counts())
print()
print("Star_Rating distribution:")
print(df['Star_Rating'].value_counts().sort_index())


## RQ1: Multiple Linear Regression — LDA Topic Weights → Star Rating

**Research Question:** Which technical feature themes (derived from LDA topic modelling) 
significantly predict citizen satisfaction (Star_Rating)?

**Method:** 
1. Run LDA (5 topics) on cleaned review text to extract per-review topic weight vectors.  
2. Label topics based on their highest-weighted terms.  
3. Fit OLS Multiple Linear Regression: `Star_Rating ~ Topic_1 + Topic_2 + Topic_3 + Topic_4 + Topic_5`  
4. Report β coefficients, standard errors, p-values, R², F-statistic, and VIF diagnostics.


In [ ]:
# ── Step 1: Fit LDA to extract 5 topic weight vectors ──────────────────────

vectorizer = CountVectorizer(
    stop_words='english',
    max_features=1000,
    min_df=2
)
dtm = vectorizer.fit_transform(df['Cleaned_Review'].fillna(''))

lda_model = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    max_iter=30,
    learning_method='batch'
)
topic_weights = lda_model.fit_transform(dtm)

# ── Step 2: Display top 12 words per topic for interpretive labelling ───────
feature_names = vectorizer.get_feature_names_out()
print("=== Top 12 Words per LDA Topic ===\n")
for i, comp in enumerate(lda_model.components_):
    top_words = [feature_names[j] for j in comp.argsort()[-12:][::-1]]
    print(f"Topic {i}: {', '.join(top_words)}")


In [ ]:
# ── Step 3: Assign interpretive labels based on dominant terms ─────────────
# Update these labels after reviewing the top-words output above.
# The mapping below reflects the topic content found in this dataset:
#   Topic 0 – Login / Authentication Issues (login, otp, error, unable)
#   Topic 1 – UX & General Experience     (app, helpful, user, experience)
#   Topic 2 – Technical Errors            (error, mobile, phone, issue, number)
#   Topic 3 – Localised / Code-switched   (hai, nahi, raha – Hindi reviews)
#   Topic 4 – Service Quality             (best, worst, service, document)

TOPIC_LABELS = [
    'Topic_LoginAuth',
    'Topic_UX_General',
    'Topic_Technical',
    'Topic_LocalLang',
    'Topic_ServiceQuality'
]

for i, label in enumerate(TOPIC_LABELS):
    df[label] = topic_weights[:, i]

print("Topic weight columns added.")
print(df[TOPIC_LABELS].describe().round(4))


In [ ]:
# ── Step 4: Multiple Linear Regression — Star_Rating ~ 5 LDA Topic Weights ─

X = df[TOPIC_LABELS]
X_const = sm.add_constant(X)        # adds intercept column
y = df['Star_Rating']

mlr_model = sm.OLS(y, X_const).fit()

print("=" * 70)
print("  RQ1 MLR Results — Predictors of Star_Rating (OLS)")
print("=" * 70)
print(mlr_model.summary())


In [ ]:
# ── Step 5: VIF Diagnostics (multicollinearity check) ───────────────────────

vif_df = pd.DataFrame({
    'Feature': TOPIC_LABELS,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print("=== Variance Inflation Factors (VIF) ===")
print("(VIF < 5 = acceptable; VIF > 10 = severe multicollinearity)")
print()
print(vif_df.to_string(index=False))


In [ ]:
os.makedirs('figures', exist_ok=True)

# ── Step 6: Visualise MLR coefficients ──────────────────────────────────────

coef_df = pd.DataFrame({
    'Topic': TOPIC_LABELS,
    'Beta': mlr_model.params[1:].values,
    'CI_lower': mlr_model.conf_int().iloc[1:, 0].values,
    'CI_upper': mlr_model.conf_int().iloc[1:, 1].values,
    'p_value': mlr_model.pvalues[1:].values
})
coef_df['Significant'] = coef_df['p_value'] < 0.05
coef_df = coef_df.sort_values('Beta')

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#e74c3c' if sig else '#95a5a6' for sig in coef_df['Significant']]
bars = ax.barh(coef_df['Topic'], coef_df['Beta'], color=colors, edgecolor='black', height=0.5)
ax.errorbar(
    coef_df['Beta'], range(len(coef_df)),
    xerr=[coef_df['Beta'] - coef_df['CI_lower'], coef_df['CI_upper'] - coef_df['Beta']],
    fmt='none', color='black', capsize=4
)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Standardised Beta Coefficient (95% CI)', fontsize=11)
ax.set_title(f'RQ1 MLR: LDA Topic Weights → Star Rating\n'
             f'R² = {mlr_model.rsquared:.3f}, Adj. R² = {mlr_model.rsquared_adj:.3f}, '
             f'F({int(mlr_model.df_model)},{int(mlr_model.df_resid)}) = {mlr_model.fvalue:.2f}, '
             f'p < .001', fontsize=10)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#e74c3c', label='p < 0.05'),
                   Patch(color='#95a5a6', label='p ≥ 0.05')], loc='lower right')
plt.tight_layout()
plt.savefig('figures/RQ1_MLR_Coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to figures/RQ1_MLR_Coefficients.png")


## RQ3: Barrier Quantification — Keyword-Coded Negative Reviews

**Method (fully replicable):**  
For each app, the subset of reviews with `Star_Rating ≤ 2` is isolated. Five barrier 
categories are defined *a priori* with explicit keyword lists. A review is coded into a 
category if **any** keyword in that category appears in `Cleaned_Review` (case-insensitive). 
The percentage reported is `count / total_negative_reviews_for_that_app × 100`.  
A review may be coded into more than one category (non-mutually exclusive).  
The keyword dictionary is printed below for transparency.


In [ ]:
# ── Barrier Keyword Dictionary (printed for transparency & replicability) ───

BARRIER_KEYWORDS = {
    'Login/Auth Issues':       ['login', 'otp', 'password', 'sign in', 'log in',
                                 'authentication', 'logging', 'unable to login'],
    'App Crashes/Technical':   ['crash', 'hang', 'freeze', 'not working', 'not open',
                                 'error', 'bug', 'glitch', 'broken'],
    'Document Access':         ['document', 'digilocker', 'unable to access',
                                 'not showing', 'not found', 'missing'],
    'Network/Loading Speed':   ['slow', 'loading', 'network', 'internet',
                                 'connection', 'server'],
    'UI/Navigation Issues':    ['confusing', 'difficult', 'hard to use',
                                 'navigation', 'interface', 'ui'],
}

print("Barrier keyword dictionary:")
for cat, kws in BARRIER_KEYWORDS.items():
    print(f"  {cat}: {kws}")


In [ ]:
# ── Compute per-app barrier frequencies and percentages ─────────────────────

neg_df = df[df['Star_Rating'] <= 2].copy()

print(f"Total negative reviews (Star ≤ 2): {len(neg_df)}")
print()
print("Negative reviews per app:")
print(neg_df['App_Name'].value_counts())
print()

rows = []
for app_name, app_grp in neg_df.groupby('App_Name'):
    n_neg = len(app_grp)
    for barrier, keywords in BARRIER_KEYWORDS.items():
        pattern = '|'.join(re.escape(k) for k in keywords)
        count = app_grp['Cleaned_Review'].str.contains(pattern, case=False, na=False).sum()
        pct = round(100 * count / n_neg, 1)
        rows.append({
            'App': app_name,
            'Barrier_Category': barrier,
            'Keyword_Method': 'Regex keyword matching on Cleaned_Review',
            'Reviews_Matched': count,
            'Total_Negatives': n_neg,
            'Pct_of_Negatives': pct
        })

barrier_df = pd.DataFrame(rows)
print("=== RQ3: Barrier Frequency Table ===")
print(barrier_df[['App','Barrier_Category','Reviews_Matched','Total_Negatives','Pct_of_Negatives']].to_string(index=False))

# Save for reporting
os.makedirs('data/processed', exist_ok=True)
barrier_df.to_csv('data/processed/RQ3_Barrier_Quantification.csv', index=False)
print("\nSaved to data/processed/RQ3_Barrier_Quantification.csv")


In [ ]:
# ── Visualise RQ3 barrier breakdown by app ───────────────────────────────────

os.makedirs('figures', exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
apps = ['DigiLocker', 'AarogyaSetu', 'UMANG']

for ax, app in zip(axes, apps):
    app_data = barrier_df[barrier_df['App'] == app].sort_values('Pct_of_Negatives', ascending=True)
    n_neg = app_data['Total_Negatives'].iloc[0]
    bars = ax.barh(app_data['Barrier_Category'], app_data['Pct_of_Negatives'],
                   color='#e74c3c', edgecolor='black', height=0.5)
    # Annotate with count/total
    for bar, (_, row) in zip(bars, app_data.iterrows()):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f"{row['Reviews_Matched']}/{n_neg}", va='center', fontsize=8)
    ax.set_xlabel('% of Negative Reviews', fontsize=10)
    ax.set_title(f'{app}\n(n={n_neg} negative reviews)', fontsize=10, fontweight='bold')
    ax.set_xlim(0, max(barrier_df['Pct_of_Negatives']) + 15)

plt.suptitle('RQ3: Adoption Barriers — % of Negative Reviews Mentioning Each Issue\n'
             '(Method: keyword matching; annotations show count/total negatives)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('figures/RQ3_Barrier_Breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to figures/RQ3_Barrier_Breakdown.png")


### ⚠️ UMANG Interpretation Note

UMANG has **31 negative reviews** out of 456 total (6.8% negative rate — the lowest of 
the three apps). Because the base is small, individual barrier percentages carry wide 
uncertainty. The figure annotations show raw counts (e.g., 2/31 = 6.5%) alongside 
percentages so the reader can assess the evidence base.  

**Do not** describe any single barrier as UMANG's "top barrier" without acknowledging this 
small-sample caveat. The correct reporting language is:  
*"Among UMANG's 31 negative reviews, the most frequently cited issues were [X] and [Y], 
each mentioned in approximately [n] reviews ([%]%). Given the small absolute count, 
these figures should be interpreted cautiously."*
